In [ ]:
pip install "gymnasium[classic-control]" stable-baselines3 sb3-contrib pyyaml pydantic

In [ ]:
pip uninstall -y torch torchvision torchaudio torchdata torchtext dgl

In [ ]:
pip install torch==2.1.2 

In [ ]:
pip install torchdata==0.7.1

In [ ]:
pip install dgl -f https://data.dgl.ai/wheels/torch-2.1/repo.html

In [ ]:
pip install "numpy<2.0.0"

In [ ]:
pip install holidays

In [36]:
from typing import Optional
import datetime
import json
import numpy as np
import pandas as pd
import gymnasium as gym
from pathlib import Path


KIND_DEMAND = 0
KIND_CAPACITY = 1


def _build_special_days(year, n_days):
    import holidays as hl
    pt_holidays = hl.country_holidays("PT", years=[year])
    start = datetime.date(year, 1, 1)
    special = set()
    for d in range(n_days):
        date = start + datetime.timedelta(days=d)
        if date.weekday() == 6 or date in pt_holidays:
            special.add(d)
    return special


class ScheduleEnvPointer(gym.Env):
    """Pointer-network MDP for the NRP.

    At each step, the agent selects an employee (or 'pass') to fill the
    current (day, shift, team) slot. Slots are processed in two passes:
      Pass 1 (DEMAND): one slot per required headcount from min_demand,
        chronological by day.
      Pass 2 (CAPACITY): extra openings per (day, shift, team) capped by
        ``capacity_slack``; they let employees accumulate toward the 223-day
        target without violating any hard constraint.
    """

    def __init__(self,
                 data_dir: str = "../../../../data/problems/SMARTASK_4TEAMS_24EMP",
                 capacity_slack: int = 2):
        super().__init__()
        base = Path(data_dir)

        with open(base / "problem.json") as f:
            prob = json.load(f)

        self.num_days = prob["temporalScope"]["numDays"]
        self.year = prob["temporalScope"]["year"]
        employees = prob["employees"]["simple"]
        self.num_employees = len(employees)
        self.employee_teams = [set(emp.get("teams", [])) for emp in employees]
        self.dual_team = [len(t) > 1 for t in self.employee_teams]

        shifts_sorted = sorted(prob["demand"]["shifts"], key=lambda s: s["order"])
        self.shift_codes = [s["code"] for s in shifts_sorted]
        self.shift_idx = {c: i for i, c in enumerate(self.shift_codes)}
        self.shift_order_map = {s["code"]: s["order"] for s in shifts_sorted}
        self.teams = list(prob["demand"]["organizationalUnits"]["teams"])
        self.team_idx = {t: i for i, t in enumerate(self.teams)}
        self.num_shifts = len(self.shift_codes)
        self.num_teams = len(self.teams)

        self.team_sizes = {
            team: sum(1 for s in self.employee_teams if team in s)
            for team in self.teams
        }
        # Pre-compute which employees belong to which team for fast eligibility.
        self.team_to_emps = {
            team: np.array([e for e in range(self.num_employees) if team in self.employee_teams[e]], dtype=int)
            for team in self.teams
        }

        vac_df = pd.read_csv(base / "vacations.csv", header=None)
        self.vac_mask = vac_df.iloc[:, 1:].values.astype(bool)

        dem_df = pd.read_csv(base / "demand.csv")
        dem_df["date"] = pd.to_datetime(dem_df["date"])
        start_ts = pd.Timestamp(f"{self.year}-01-01")
        dem_df["day_idx"] = (dem_df["date"] - start_ts).dt.days

        self.min_demand = np.zeros((self.num_days, self.num_shifts, self.num_teams), dtype=int)
        for _, row in dem_df.iterrows():
            d = int(row["day_idx"])
            s = self.shift_idx[row["shift"]]
            t = self.team_idx[row["team"]]
            self.min_demand[d, s, t] = int(row["minimum"])

        self.special_days = _build_special_days(self.year, self.num_days)

        self.max_days_per_year = 223
        self.max_consecutive_days = 5
        self.special_days_cap = 22
        self.capacity_slack = capacity_slack

        # Action space: num_employees points (one per employee) plus one PASS action.
        self.NUM_ACTIONS = self.num_employees + 1
        self.PASS_ACTION = self.num_employees

        self.action_space = gym.spaces.Discrete(self.NUM_ACTIONS)
        # Observation is opaque; the model consumes structured tensors instead.
        self.observation_space = gym.spaces.Box(low=0.0, high=1.0, shape=(1,), dtype=np.float32)

        self.reset()

    # ------------------------------------------------------------------
    # Slot queue
    # ------------------------------------------------------------------

    def _build_slot_queue(self):
        """DEMAND first (chronological by day), then CAPACITY.

        Chronological-by-day keeps the prev/next-day shift-order check
        well-defined: when we process DEMAND on day d, day d-1 is already
        finalised; day d+1 is empty. The mask handles both directions.

        Within a day we iterate (shift, team) in shift-order to favour
        morning before afternoon, which preserves shift-order legality
        for whichever employee gets picked.
        """
        demand = []
        for d in range(self.num_days):
            for s_idx in range(self.num_shifts):
                for t_idx in range(self.num_teams):
                    headcount = int(self.min_demand[d, s_idx, t_idx])
                    for _ in range(headcount):
                        demand.append((d, s_idx, t_idx, KIND_DEMAND))

        capacity = []
        for d in range(self.num_days):
            for s_idx in range(self.num_shifts):
                for t_idx in range(self.num_teams):
                    for _ in range(self.capacity_slack):
                        capacity.append((d, s_idx, t_idx, KIND_CAPACITY))

        return demand + capacity, len(demand)

    # ------------------------------------------------------------------
    # Eligibility / mask
    # ------------------------------------------------------------------

    def _emp_streak_if_works(self, emp, day):
        """Hypothetical consecutive-day streak if ``emp`` works on ``day``."""
        streak = 1
        d = day - 1
        while d >= 0 and self.emp_day_shift[emp, d] >= 0:
            streak += 1
            d -= 1
        d = day + 1
        while d < self.num_days and self.emp_day_shift[emp, d] >= 0:
            streak += 1
            d += 1
        return streak

    def _emp_can_fill(self, emp, day, s_idx, t_idx):
        # Team membership
        if self.teams[t_idx] not in self.employee_teams[emp]:
            return False
        # Vacation
        if self.vac_mask[emp, day]:
            return False
        # Already assigned a shift on this day
        if self.emp_day_shift[emp, day] >= 0:
            return False
        # Yearly cap
        if self.days_worked[emp] >= self.max_days_per_year:
            return False
        # Special-day cap
        if day in self.special_days and self.special_days_worked[emp] >= self.special_days_cap:
            return False
        # Consecutive streak
        if self._emp_streak_if_works(emp, day) > self.max_consecutive_days:
            return False
        # Shift-order: yesterday's shift cannot have higher order than today's.
        if day > 0:
            prev = self.emp_day_shift[emp, day - 1]
            if prev >= 0:
                prev_order = self.shift_order_map[self.shift_codes[prev]]
                today_order = self.shift_order_map[self.shift_codes[s_idx]]
                if prev_order > today_order:
                    return False
        # Shift-order: tomorrow's shift cannot have lower order than today's.
        if day < self.num_days - 1:
            nxt = self.emp_day_shift[emp, day + 1]
            if nxt >= 0:
                nxt_order = self.shift_order_map[self.shift_codes[nxt]]
                today_order = self.shift_order_map[self.shift_codes[s_idx]]
                if today_order > nxt_order:
                    return False
        return True

    def get_employee_mask(self):
        """Bool mask of shape [num_employees + 1].
        Indices 0..num_employees-1: True iff that employee can legally fill
        the current slot. Index PASS_ACTION (=num_employees): True iff the
        agent is allowed to leave the slot uncovered (always for CAPACITY
        slots; only as a forced fallback for DEMAND slots)."""
        mask = np.zeros(self.NUM_ACTIONS, dtype=bool)
        if self.slot_idx >= len(self.slot_queue):
            mask[self.PASS_ACTION] = True
            return mask
        d, s_idx, t_idx, kind = self.slot_queue[self.slot_idx]
        for e in self.team_to_emps[self.teams[t_idx]]:
            if self._emp_can_fill(int(e), d, s_idx, t_idx):
                mask[int(e)] = True
        if kind == KIND_CAPACITY:
            mask[self.PASS_ACTION] = True
        else:
            # DEMAND: PASS only when no employee is eligible (silent-failure fallback).
            if not mask[:self.num_employees].any():
                mask[self.PASS_ACTION] = True
        return mask

    # ------------------------------------------------------------------
    # Current step accessors (used by the model wrapper)
    # ------------------------------------------------------------------

    def current_slot(self):
        if self.slot_idx >= len(self.slot_queue):
            return None
        return self.slot_queue[self.slot_idx]

    def current_day(self):
        s = self.current_slot()
        return s[0] if s is not None else 0

    # ------------------------------------------------------------------
    # Reward
    # ------------------------------------------------------------------

    def _team_balance_bonus(self, emp_id, team):
        if not self.dual_team[emp_id]:
            return 0.0
        sizes = {t: self.team_sizes[t] for t in self.employee_teams[emp_id]}
        min_size, max_size = min(sizes.values()), max(sizes.values())
        if min_size == max_size:
            return 0.0
        smaller = {t for t, s in sizes.items() if s == min_size}
        return (max_size - min_size) * 0.5 if team in smaller else 0.0

    def _step_reward(self, action, d, s_idx, t_idx, kind):
        if action == self.PASS_ACTION:
            # DEMAND pass = an unfilled slot. CAPACITY pass = legal idle.
            return -5.0 if kind == KIND_DEMAND else 0.0
        team = self.teams[t_idx]
        bonus = self._team_balance_bonus(action, team)
        if kind == KIND_DEMAND:
            return 1.0 + 0.2 * bonus
        # CAPACITY: positive but smaller; diminish near the 223-day cap.
        if self.days_worked[action] >= 200:
            return 0.25 + 0.1 * bonus
        return 0.5 + 0.1 * bonus

    # ------------------------------------------------------------------
    # Lifecycle
    # ------------------------------------------------------------------

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.days_worked = np.zeros(self.num_employees, dtype=int)
        self.special_days_worked = np.zeros(self.num_employees, dtype=int)
        # emp_day_shift[e, d]: shift_idx if working, -1 if not. emp_day_team holds the team.
        self.emp_day_shift = np.full((self.num_employees, self.num_days), -1, dtype=int)
        self.emp_day_team = np.full((self.num_employees, self.num_days), -1, dtype=int)
        self.daily_coverage = np.zeros((self.num_days, self.num_shifts, self.num_teams), dtype=np.float32)
        self.slot_queue, self.num_demand_slots = self._build_slot_queue()
        self.slot_idx = 0
        self.demand_pass_count = 0
        # Bookkeeping for the training loop.
        self.demand_shortfall_at_phase_end = 0
        return self._obs(), {}

    def _obs(self):
        return np.zeros(1, dtype=np.float32)

    def step(self, action):
        if self.slot_idx >= len(self.slot_queue):
            return self._obs(), 0.0, True, False, {}
        d, s_idx, t_idx, kind = self.slot_queue[self.slot_idx]
        action = int(action)

        if action != self.PASS_ACTION:
            assert 0 <= action < self.num_employees, f"invalid action {action}"
            self.emp_day_shift[action, d] = s_idx
            self.emp_day_team[action, d] = t_idx
            self.days_worked[action] += 1
            self.daily_coverage[d, s_idx, t_idx] += 1
            if d in self.special_days:
                self.special_days_worked[action] += 1
        elif kind == KIND_DEMAND:
            self.demand_pass_count += 1

        reward = self._step_reward(action, d, s_idx, t_idx, kind)

        self.slot_idx += 1

        # Capture shortfall at the demand/capacity boundary for diagnostics.
        if self.slot_idx == self.num_demand_slots:
            self.demand_shortfall_at_phase_end = int(
                np.maximum(0, self.min_demand - self.daily_coverage).sum()
            )

        terminated = self.slot_idx >= len(self.slot_queue)
        if terminated:
            shortfall = int(np.maximum(0, self.min_demand - self.daily_coverage).sum())
            if shortfall == 0:
                reward += 200.0
        return self._obs(), reward, terminated, False, {}

    # ------------------------------------------------------------------
    # Rendering / inspection
    # ------------------------------------------------------------------

    def action_label_for(self, emp, day):
        s = self.emp_day_shift[emp, day]
        if s < 0:
            return "-"
        t = self.emp_day_team[emp, day]
        return f"{self.shift_codes[s]}-{self.teams[t]}"

    def render_schedule(self):
        return [
            [self.action_label_for(emp, day) for day in range(self.num_days)]
            for emp in range(self.num_employees)
        ]


# Backward-compatible alias so downstream cells can reference ScheduleEnv.
ScheduleEnv = ScheduleEnvPointer


In [ ]:
import dgl
import dgl.nn as dglnn
import torch
import numpy as np


def emp_feat_dim(env):
    # [days_worked/223, streak/5, in_team flag per team, emp_position]
    return 2 + env.num_teams + 1


def day_feat_dim(env):
    # [cov_gap per (team, shift), is_special, day_position]
    return env.num_shifts * env.num_teams + 2


def build_graph(env):
    sources, destinations = [], []
    for emp in range(env.num_employees):
        for day in range(env.num_days):
            sources.append(emp)
            destinations.append(day)
    graph_data = {
        ("employee", "assigned_to", "day"): (torch.tensor(sources), torch.tensor(destinations)),
        ("day", "staffed_by", "employee"): (torch.tensor(destinations), torch.tensor(sources)),
    }
    g = dgl.heterograph(graph_data)
    update_graph_features(g, env)
    return g


def update_graph_features(g, env):
    emp_feats = np.zeros((env.num_employees, emp_feat_dim(env)), dtype=np.float32)
    for emp in range(env.num_employees):
        emp_feats[emp, 0] = env.days_worked[emp] / 223.0
        emp_feats[emp, 1] = env._consecutive_streak_if_work(emp, env.current_day()) / 5.0
        for t_idx, team in enumerate(env.teams):
            emp_feats[emp, 2 + t_idx] = float(team in env.employee_teams[emp])
        emp_feats[emp, 2 + env.num_teams] = emp / env.num_employees

    day_feats = np.zeros((env.num_days, day_feat_dim(env)), dtype=np.float32)
    for d in range(env.num_days):
        idx = 0
        for t_idx in range(env.num_teams):
            for s_idx in range(env.num_shifts):
                day_feats[d, idx] = env.min_demand[d, s_idx, t_idx] - env.daily_coverage[d, s_idx, t_idx]
                idx += 1
        day_feats[d, idx] = float(d in env.special_days)
        day_feats[d, idx + 1] = d / env.num_days

    g.nodes["employee"].data["feat"] = torch.tensor(emp_feats, dtype=torch.float32)
    g.nodes["day"].data["feat"] = torch.tensor(day_feats, dtype=torch.float32)


def coverage_gap_vector(env, day_id):
    gaps = []
    for t_idx in range(env.num_teams):
        for s_idx in range(env.num_shifts):
            gaps.append(env.min_demand[day_id, s_idx, t_idx] - env.daily_coverage[day_id, s_idx, t_idx])
    return gaps


In [38]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class PointerActorCritic(nn.Module):
    """GNN encoder (heterogeneous SAGE) + pointer-attention head over employees.

    Per step the head receives:
      - emp_emb     [num_emp, D]   from the GNN trunk
      - day_emb     [num_days, D]  from the GNN trunk (we index into it)
      - day_ids     [N]            which day this step is at
      - slot_ctx    [N, ctx_dim]   shift one-hot, team one-hot, kind one-hot
      - emp_dyn     [N, num_emp, 2] live per-emp features (days_worked, streak)
      - mask        [N, num_emp+1] eligibility mask

    It produces:
      - probs [N, num_emp+1]   softmax over (employees, PASS)
      - value [N]              critic estimate
    """

    def __init__(self, emp_in_feats, day_in_feats, num_shifts, num_teams,
                 num_employees, hidden_dim=64, encoded_dim=64):
        super().__init__()
        self.encoded_dim = encoded_dim
        self.num_shifts = num_shifts
        self.num_teams = num_teams
        self.num_employees = num_employees

        self.emp_proj = nn.Linear(emp_in_feats, hidden_dim)
        self.day_proj = nn.Linear(day_in_feats, hidden_dim)

        self.conv1 = dglnn.HeteroGraphConv({
            'assigned_to': dglnn.SAGEConv(hidden_dim, hidden_dim, 'mean'),
            'staffed_by': dglnn.SAGEConv(hidden_dim, hidden_dim, 'mean'),
        }, aggregate='sum')

        self.conv2 = dglnn.HeteroGraphConv({
            'assigned_to': dglnn.SAGEConv(hidden_dim, encoded_dim, 'mean'),
            'staffed_by': dglnn.SAGEConv(hidden_dim, encoded_dim, 'mean'),
        }, aggregate='sum')

        # Slot context = day_emb (encoded_dim) + shift one-hot + team one-hot + kind one-hot (2).
        slot_ctx_dim = encoded_dim + num_shifts + num_teams + 2
        self.slot_ctx_dim = slot_ctx_dim

        self.query_proj = nn.Sequential(
            nn.Linear(slot_ctx_dim, 64),
            nn.Tanh(),
            nn.Linear(64, encoded_dim),
        )
        # Keys derived from emp_emb (encoded_dim) + per-emp dyn feats (2 — normalised).
        self.key_proj = nn.Sequential(
            nn.Linear(encoded_dim + 2, 64),
            nn.Tanh(),
            nn.Linear(64, encoded_dim),
        )
        # Pass logit conditioned on slot context only.
        self.pass_logit = nn.Sequential(
            nn.Linear(slot_ctx_dim, 32),
            nn.Tanh(),
            nn.Linear(32, 1),
        )
        # Value: mean-pooled emp_emb concatenated with slot context.
        self.value_head = nn.Sequential(
            nn.Linear(encoded_dim + slot_ctx_dim, 64),
            nn.Tanh(),
            nn.Linear(64, 1),
        )

    @classmethod
    def from_env(cls, env, hidden_dim=64, encoded_dim=64):
        return cls(
            emp_in_feats=emp_feat_dim(env),
            day_in_feats=day_feat_dim(env),
            num_shifts=env.num_shifts,
            num_teams=env.num_teams,
            num_employees=env.num_employees,
            hidden_dim=hidden_dim,
            encoded_dim=encoded_dim,
        )

    def gnn_forward(self, g):
        h = {
            "employee": self.emp_proj(g.nodes["employee"].data["feat"]),
            "day": self.day_proj(g.nodes["day"].data["feat"]),
        }
        h = self.conv1(g, h)
        h = {k: F.relu(v) for k, v in h.items()}
        h = self.conv2(g, h)
        h = {k: F.relu(v) for k, v in h.items()}
        return h["employee"], h["day"]

    def _heads(self, emp_emb, day_emb, day_ids, slot_ctx, emp_dyn, mask):
        """
        emp_emb:  [num_emp, D]  (single-graph case) OR [N, num_emp, D]
        day_emb:  [num_days, D]
        day_ids:  [N]
        slot_ctx: [N, num_shifts + num_teams + 2]    # shift_oh ⊕ team_oh ⊕ kind_oh
        emp_dyn:  [N, num_emp, 2]                    # normalised (days_worked/223, streak/5)
        mask:     [N, num_emp + 1]
        """
        N = day_ids.shape[0]
        D = self.encoded_dim

        cur_day_emb = day_emb[day_ids]                        # [N, D]
        full_slot_ctx = torch.cat([cur_day_emb, slot_ctx], dim=-1)  # [N, slot_ctx_dim]

        # Query
        query = self.query_proj(full_slot_ctx)                # [N, D]

        # Keys: broadcast emp_emb if needed
        if emp_emb.dim() == 2:
            emp_emb_b = emp_emb.unsqueeze(0).expand(N, -1, -1)  # [N, num_emp, D]
        else:
            emp_emb_b = emp_emb
        key_in = torch.cat([emp_emb_b, emp_dyn], dim=-1)        # [N, num_emp, D+2]
        keys = self.key_proj(key_in)                            # [N, num_emp, D]

        # Scaled dot-product attention scores
        scores = (keys * query.unsqueeze(1)).sum(dim=-1) / (D ** 0.5)   # [N, num_emp]

        # Pass logit per step
        pass_l = self.pass_logit(full_slot_ctx)                 # [N, 1]
        logits = torch.cat([scores, pass_l], dim=-1)            # [N, num_emp + 1]

        mask_b = mask.bool()
        # Safety: if mask is somehow empty, allow PASS.
        empty = ~mask_b.any(dim=-1)
        if empty.any():
            mask_b = mask_b.clone()
            mask_b[empty, -1] = True
        logits = logits.masked_fill(~mask_b, float("-inf"))
        probs = F.softmax(logits, dim=-1)

        # Value head
        pooled = emp_emb_b.mean(dim=1)                          # [N, D]
        v_in = torch.cat([pooled, full_slot_ctx], dim=-1)
        values = self.value_head(v_in).squeeze(-1)              # [N]

        return probs, values

    def forward(self, g, day_ids, slot_ctx, emp_dyn, mask):
        emp_emb, day_emb = self.gnn_forward(g)
        return self._heads(emp_emb, day_emb, day_ids, slot_ctx, emp_dyn, mask)


def build_slot_ctx_tensor(env, slot):
    """[num_shifts + num_teams + 2] tensor: shift one-hot, team one-hot, kind one-hot."""
    d, s_idx, t_idx, kind = slot
    v = np.zeros(env.num_shifts + env.num_teams + 2, dtype=np.float32)
    v[s_idx] = 1.0
    v[env.num_shifts + t_idx] = 1.0
    v[env.num_shifts + env.num_teams + kind] = 1.0
    return v


In [39]:
from dataclasses import dataclass, field
from torch.distributions import Categorical


@dataclass
class Trajectory:
    day_ids: list = field(default_factory=list)
    slot_ctxs: list = field(default_factory=list)
    emp_dyns: list = field(default_factory=list)
    masks: list = field(default_factory=list)
    actions: list = field(default_factory=list)
    log_probs_old: list = field(default_factory=list)
    rewards: list = field(default_factory=list)
    values: list = field(default_factory=list)
    # Per-step graph snapshots — only one unique per day, but stored per step
    # for easy minibatch lookup. emp_feats_snapshots[t] / day_feats_snapshots[t]
    # are the inputs the GNN saw when this step was decided.
    emp_feats_snapshots: list = field(default_factory=list)
    day_feats_snapshots: list = field(default_factory=list)

    def to_tensors(self):
        return {
            "day_ids": torch.tensor(self.day_ids, dtype=torch.long),
            "slot_ctxs": torch.stack(self.slot_ctxs),
            "emp_dyns": torch.stack(self.emp_dyns),
            "masks": torch.stack(self.masks),
            "actions": torch.tensor(self.actions, dtype=torch.long),
            "log_probs_old": torch.tensor(self.log_probs_old, dtype=torch.float32),
            "rewards": torch.tensor(self.rewards, dtype=torch.float32),
            "values": torch.tensor(self.values, dtype=torch.float32),
            "emp_feats": torch.stack(self.emp_feats_snapshots),
            "day_feats": torch.stack(self.day_feats_snapshots),
        }


def _normalise_dyn(env, dyn):
    """[num_emp, 2] -> normalise (days_worked/223, streak/5)."""
    out = dyn.copy()
    out[:, 0] /= 223.0
    out[:, 1] /= 5.0
    return out


def collect_trajectory(env, model, graph):
    model.eval()
    traj = Trajectory()
    env.reset()
    terminated = truncated = False

    cached_day = None
    cached_emp_emb = None
    cached_day_emb = None
    cached_emp_feats = None
    cached_day_feats = None

    with torch.no_grad():
        while not (terminated or truncated):
            slot = env.current_slot()
            if slot is None:
                break
            day_id = slot[0]

            # GNN snapshot is per-day. We refresh the graph features and rerun
            # the trunk only when the day changes.
            if day_id != cached_day:
                update_graph_features(graph, env)
                cached_emp_emb, cached_day_emb = model.gnn_forward(graph)
                cached_emp_feats = graph.nodes["employee"].data["feat"].clone()
                cached_day_feats = graph.nodes["day"].data["feat"].clone()
                cached_day = day_id

            slot_ctx_np = build_slot_ctx_tensor(env, slot)
            emp_dyn_np = _normalise_dyn(env, emp_dyn_feats_all(env))
            mask_np = env.get_employee_mask()

            day_id_t = torch.tensor([day_id], dtype=torch.long)
            slot_ctx_t = torch.tensor(slot_ctx_np, dtype=torch.float32).unsqueeze(0)
            emp_dyn_t = torch.tensor(emp_dyn_np, dtype=torch.float32).unsqueeze(0)
            mask_t = torch.tensor(mask_np, dtype=torch.bool).unsqueeze(0)

            probs, values = model._heads(
                cached_emp_emb, cached_day_emb,
                day_id_t, slot_ctx_t, emp_dyn_t, mask_t,
            )

            dist = Categorical(probs=probs[0])
            action = dist.sample()

            _, reward, terminated, truncated, _ = env.step(action.item())

            traj.day_ids.append(day_id)
            traj.slot_ctxs.append(slot_ctx_t[0])
            traj.emp_dyns.append(emp_dyn_t[0])
            traj.masks.append(mask_t[0])
            traj.actions.append(action.item())
            traj.log_probs_old.append(dist.log_prob(action).item())
            traj.rewards.append(float(reward))
            traj.values.append(values[0].item())
            traj.emp_feats_snapshots.append(cached_emp_feats)
            traj.day_feats_snapshots.append(cached_day_feats)

    return traj


In [40]:
def compute_gae(rewards, values, gamma=0.99, lam=0.95):
    T = len(rewards)
    rewards_a = np.array(rewards, dtype=np.float32)
    values_a = np.array(values, dtype=np.float32)
    values_ext = np.append(values_a, 0.0)

    advantages = np.zeros(T, dtype=np.float32)
    gae = 0.0
    for t in reversed(range(T)):
        delta = rewards_a[t] + gamma * values_ext[t + 1] - values_ext[t]
        gae = delta + gamma * lam * gae
        advantages[t] = gae

    returns = advantages + values_a
    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-5)

    return (
        torch.tensor(advantages, dtype=torch.float32),
        torch.tensor(returns, dtype=torch.float32),
    )


In [43]:
from torch.utils.data.sampler import BatchSampler, SubsetRandomSampler


def ppo_update(model, optimizer, batch, advantages, returns, graph,
               clip_eps=0.2, value_coeff=0.5, entropy_coeff=0.01,
               K_epochs=4, mini_batch_size=512):
    T = batch["actions"].shape[0]
    losses = []
    track_entropy = []
    model.train()

    for _ in range(K_epochs):
        for index in BatchSampler(SubsetRandomSampler(range(T)), mini_batch_size, False):
            idx = torch.tensor(index)

            # Restore graph features from snapshot and run GNN
            graph.nodes["employee"].data["feat"] = batch["emp_feats"][idx[0]]
            graph.nodes["day"].data["feat"] = batch["day_feats"][idx[0]]
            emp_emb, day_emb = model.gnn_forward(graph)

            # Recompute action probabilities and state values for the sampled batch using the current model parameters and the cached GNN embeddings
            probs_new, values_new = model._heads(
                emp_emb, day_emb,
                batch["day_ids"][idx],
                batch["slot_ctxs"][idx],
                batch["emp_dyns"][idx],
                batch["masks"][idx],
            )

            dist_now = Categorical(probs=probs_new)
            a_logprob_now = dist_now.log_prob(batch["actions"][idx])
            dist_entropy = dist_now.entropy()

            ratios = torch.exp(a_logprob_now - batch["log_probs_old"][idx])
            adv = advantages[idx]
            surr1 = ratios * adv
            surr2 = torch.clamp(ratios, 1.0 - clip_eps, 1.0 + clip_eps) * adv
            actor_loss = -torch.min(surr1, surr2).mean() - entropy_coeff * dist_entropy.mean()

            critic_loss = F.smooth_l1_loss(values_new, returns[idx])

            loss = actor_loss + value_coeff * critic_loss

            track_entropy.append(dist_entropy.mean().item())

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
            optimizer.step()
            losses.append(loss.item())

    return float(np.mean(losses)), float(np.mean(track_entropy))


In [44]:
import os

NUM_EPISODES = 10000
GAMMA = 0.99
LAM = 0.95
CLIP_EPS = 0.15
VALUE_COEFF = 0.5
ENTROPY_COEFF_0 = 0.05
ENTROPY_MIN = 0.005
K_EPOCHS = 3
MINI_BATCH_SIZE = 512
LR = 3e-4
SAVE_EVERY = 50

BEST_CKPT = "best_4teams_pointer_v2.pth"
LATEST_CKPT = "latest_4teams_pointer_v2.pth"
RESUME_FROM = LATEST_CKPT  # set to None to start fresh

env = ScheduleEnvPointer()
graph = build_graph(env)
model = PointerActorCritic.from_env(env)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, eps=1e-5)

print(f"Scenario: {env.num_teams} teams ({env.teams}), {env.num_shifts} shifts ({env.shift_codes}), "
      f"{env.num_employees} employees, NUM_ACTIONS={env.NUM_ACTIONS}")
print(f"Slot queue: {env.num_demand_slots} DEMAND + {len(env.slot_queue) - env.num_demand_slots} CAPACITY "
      f"= {len(env.slot_queue)} total")

best_reward = -float("inf")
best_shortfall = int(env.min_demand.sum())  # worst case (all DEMAND passed)
start_episode = 0

if RESUME_FROM and os.path.exists(RESUME_FROM):
    ckpt = torch.load(RESUME_FROM, weights_only=True)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    start_episode = ckpt["episode"]
    best_reward = float(ckpt.get("best_reward", -float("inf")))
    best_shortfall = int(ckpt.get("best_shortfall", best_shortfall))
    print(f"Resumed from episode {start_episode}, best reward = {best_reward:.1f}, best shortfall = {best_shortfall}")

for episode in range(start_episode, NUM_EPISODES):
    entropy_coeff = max(ENTROPY_MIN, ENTROPY_COEFF_0 * (0.999 ** episode))

    traj = collect_trajectory(env, model, graph)
    batch = traj.to_tensors()

    advantages, returns = compute_gae(traj.rewards, traj.values, GAMMA, LAM)

    mean_loss, mean_entropy = ppo_update(
        model, optimizer, batch, advantages, returns, graph,
        clip_eps=CLIP_EPS,
        value_coeff=VALUE_COEFF,
        entropy_coeff=entropy_coeff,
        K_epochs=K_EPOCHS,
        mini_batch_size=MINI_BATCH_SIZE,
    )

    total_reward = sum(traj.rewards)
    shortfall = int(np.maximum(0, env.min_demand - env.daily_coverage).sum())
    demand_passes = env.demand_pass_count
    mean_days = env.days_worked.mean()
    days_worked_str = ",".join(str(int(d)) for d in env.days_worked)

    improved = False
    if shortfall < best_shortfall or (shortfall == best_shortfall and total_reward > best_reward):
        best_shortfall = int(shortfall)
        best_reward = float(total_reward)
        improved = True
        torch.save(
            {"episode": episode, "model_state_dict": model.state_dict(),
             "optimizer_state_dict": optimizer.state_dict(),
             "best_reward": best_reward, "best_shortfall": best_shortfall},
            BEST_CKPT,
        )
        print(f"  NEW BEST: R={best_reward:.1f}, shortfall={best_shortfall} at ep {episode} "
              f"| demand_pass={demand_passes} | days={mean_days:.0f}")

    if episode % 10 == 0:
        print(f"Ep {episode:>4} | R={total_reward:>8.1f} | Best={best_reward:>8.1f} "
              f"| Loss={mean_loss:.4f} | ent={mean_entropy:.4f} "
              f"| shortfall={shortfall} (pass={demand_passes}) "
              f"| days={mean_days:.0f} [{days_worked_str}]")

    if (episode + 1) % SAVE_EVERY == 0:
        torch.save(
            {"episode": episode + 1, "model_state_dict": model.state_dict(),
             "optimizer_state_dict": optimizer.state_dict(),
             "best_reward": best_reward, "best_shortfall": best_shortfall},
            LATEST_CKPT,
        )


Scenario: 4 teams (['A', 'B', 'C', 'D']), 2 shifts (['M', 'T']), 24 employees, NUM_ACTIONS=25
Slot queue: 2920 DEMAND + 5840 CAPACITY = 8760 total
  NEW BEST: R=3961.8, shortfall=5 at ep 0 | demand_pass=5 | days=223
Ep    0 | R=  3961.8 | Best=  3961.8 | Loss=2.6367 | ent=0.9502 | shortfall=5 (pass=5) | days=223 [223,223,223,223,223,223,223,223,220,223,223,223,223,223,223,223,223,223,223,223,215,223,223,223]
  NEW BEST: R=3970.0, shortfall=4 at ep 3 | demand_pass=4 | days=223
  NEW BEST: R=3975.5, shortfall=3 at ep 6 | demand_pass=3 | days=223
Ep   10 | R=  3970.0 | Best=  3975.5 | Loss=2.5029 | ent=0.8459 | shortfall=4 (pass=4) | days=223 [223,223,223,223,223,223,223,223,223,223,223,223,223,223,223,223,223,223,223,223,223,223,223,223]
  NEW BEST: R=4186.2, shortfall=0 at ep 14 | demand_pass=0 | days=222
Ep   20 | R=  3959.5 | Best=  4186.2 | Loss=2.2982 | ent=0.8906 | shortfall=5 (pass=5) | days=222 [223,223,223,223,214,223,223,223,215,223,223,223,223,223,223,223,223,223,223,223,220,2

KeyboardInterrupt: 

In [45]:
def evaluate(model, env, graph, greedy=False):
    model.eval()
    env.reset()
    terminated = truncated = False
    total_reward = 0.0

    cached_day = None
    cached_emp_emb = None
    cached_day_emb = None

    with torch.no_grad():
        while not (terminated or truncated):
            slot = env.current_slot()
            if slot is None:
                break
            day_id = slot[0]

            if day_id != cached_day:
                update_graph_features(graph, env)
                cached_emp_emb, cached_day_emb = model.gnn_forward(graph)
                cached_day = day_id

            slot_ctx_np = build_slot_ctx_tensor(env, slot)
            emp_dyn_np = _normalise_dyn(env, emp_dyn_feats_all(env))
            mask_np = env.get_employee_mask()

            day_id_t = torch.tensor([day_id], dtype=torch.long)
            slot_ctx_t = torch.tensor(slot_ctx_np, dtype=torch.float32).unsqueeze(0)
            emp_dyn_t = torch.tensor(emp_dyn_np, dtype=torch.float32).unsqueeze(0)
            mask_t = torch.tensor(mask_np, dtype=torch.bool).unsqueeze(0)

            probs, _ = model._heads(
                cached_emp_emb, cached_day_emb,
                day_id_t, slot_ctx_t, emp_dyn_t, mask_t,
            )

            action = probs[0].argmax() if greedy else Categorical(probs=probs[0]).sample()
            _, reward, terminated, truncated, _ = env.step(action.item())
            total_reward += reward

    shortfall = int(np.maximum(0, env.min_demand - env.daily_coverage).sum())

    snapshot = {
        "demand_pass_count": env.demand_pass_count,
        "demand_shortfall_at_phase_end": env.demand_shortfall_at_phase_end,
        "daily_coverage": env.daily_coverage.copy(),
    }
    schedule = env.render_schedule()
    return total_reward, shortfall, env.days_worked.tolist(), schedule, snapshot


ckpt = torch.load(BEST_CKPT, weights_only=True)
model.load_state_dict(ckpt["model_state_dict"])
print(f"Checkpoint: episode {ckpt['episode']}, best reward = {ckpt['best_reward']:.1f}, "
      f"best shortfall = {ckpt.get('best_shortfall', 'n/a')}\n")

N = 20
results = []
for i in range(N):
    np.random.seed(42 + i)
    reward, shortfall, days_worked, schedule, snap = evaluate(model, env, graph, greedy=False)
    results.append((reward, shortfall, days_worked, schedule, snap))
    print(f"Run {i+1:2d} | Reward: {reward:8.1f} | Shortfall: {shortfall:3d} "
          f"| demand_pass: {snap['demand_pass_count']:3d} "
          f"| Days worked: {days_worked} (avg={np.mean(days_worked):.0f})")

best_idx = min(range(N), key=lambda i: results[i][1])
best_reward, best_shortfall, best_days, best_schedule, best_snap = results[best_idx]

print(f"\n{'='*60}")
print(f"BEST: Run {best_idx+1} | Reward: {best_reward:.1f} | Shortfall: {best_shortfall}")
print(f"Demand passes: {best_snap['demand_pass_count']}")
print(f"Days worked/employee: {best_days} (avg={np.mean(best_days):.0f})")
print(f"{'='*60}\n")

best_coverage = best_snap["daily_coverage"]
for d in range(env.num_days):
    gaps = []
    for s_idx, shift in enumerate(env.shift_codes):
        for t_idx, team in enumerate(env.teams):
            gap = int(env.min_demand[d, s_idx, t_idx] - best_coverage[d, s_idx, t_idx])
            if gap > 0:
                gaps.append(f"{shift}-{team}={gap}")
    if gaps:
        print(f"Day {d:3d}: {', '.join(gaps)}")

print()
for emp in range(len(best_schedule)):
    print(f"Employee {emp+1:2d}: {best_schedule[emp]}")

import csv
schedule_csv = f"best_schedule_{env.num_teams}teams_pointer_v2.csv"
with open(schedule_csv, "w", newline="") as f:
    writer = csv.writer(f)
    header = ["Employee"] + [f"Day_{d}" for d in range(len(best_schedule[0]))]
    writer.writerow(header)
    for emp in range(len(best_schedule)):
        writer.writerow([f"Employee_{emp+1}"] + best_schedule[emp])

print(f"\nBest schedule written to {schedule_csv}")


/home/joao/.local/lib/python3.11/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


Checkpoint: episode 14, best reward = 4186.2, best shortfall = 0

Run  1 | Reward:   3966.0 | Shortfall:   4 | demand_pass:   4 | Days worked: [223, 223, 223, 223, 219, 223, 223, 223, 214, 223, 223, 223, 223, 223, 223, 223, 220, 223, 223, 223, 223, 223, 223, 223] (avg=222)
Run  2 | Reward:   3963.0 | Shortfall:   5 | demand_pass:   5 | Days worked: [223, 223, 223, 223, 218, 223, 223, 223, 222, 223, 223, 223, 223, 223, 223, 223, 223, 223, 223, 223, 223, 223, 223, 223] (avg=223)
Run  3 | Reward:   3980.2 | Shortfall:   2 | demand_pass:   2 | Days worked: [223, 223, 223, 223, 220, 223, 223, 223, 223, 223, 223, 223, 223, 223, 223, 223, 223, 223, 223, 223, 223, 223, 223, 223] (avg=223)
Run  4 | Reward:   3963.2 | Shortfall:   5 | demand_pass:   5 | Days worked: [223, 223, 223, 223, 219, 223, 223, 223, 223, 223, 223, 223, 223, 223, 223, 223, 222, 223, 223, 223, 223, 223, 223, 223] (avg=223)
Run  5 | Reward:   3974.8 | Shortfall:   3 | demand_pass:   3 | Days worked: [223, 223, 223, 223, 223,

In [ ]:
import matplotlib.pyplot as plt

ckpt = torch.load(BEST_CKPT, weights_only=True)
model.load_state_dict(ckpt["model_state_dict"])
_, _, days_worked_list, _, snap = evaluate(model, env, graph, greedy=False)

days   = np.arange(env.num_days)
cov    = snap["daily_coverage"]
demand = env.min_demand

fig, axes = plt.subplots(
    env.num_shifts, env.num_teams,
    figsize=(4 * env.num_teams, 4 * env.num_shifts),
    sharex=True, sharey=True, squeeze=False,
)
for s_idx, shift in enumerate(env.shift_codes):
    for t_idx, team in enumerate(env.teams):
        ax = axes[s_idx][t_idx]
        ax.plot(days, cov[:, s_idx, t_idx], label='Coverage', alpha=0.7)
        ax.plot(days, demand[:, s_idx, t_idx], label='Demand', alpha=0.7, linestyle='--', color='r')
        ax.fill_between(
            days, cov[:, s_idx, t_idx], demand[:, s_idx, t_idx],
            where=cov[:, s_idx, t_idx] < demand[:, s_idx, t_idx],
            color='red', alpha=0.2, label='Shortfall',
        )
        ax.set_title(f"{shift}-{team}")
        if t_idx == 0:
            ax.set_ylabel('Employees')
        if s_idx == env.num_shifts - 1:
            ax.set_xlabel('Day of year')
        ax.legend(loc='upper right', fontsize=8)

fig.suptitle('Coverage vs Demand across the year (pointer-net v2)', fontsize=14)
plt.tight_layout()
plt.show()

pair_labels = [f"{s}-{t}" for t in env.teams for s in env.shift_codes]
header = f"{'Quarter':<12} " + " ".join(f"{p:>8}" for p in pair_labels) + f" {'Total':>6}"
print()
print(header)
print("-" * len(header))
total_viol = 0
for q, (start, end) in enumerate([(0, 91), (91, 182), (182, 273), (273, 365)], 1):
    per_pair = []
    q_total = 0
    for team in env.teams:
        for shift in env.shift_codes:
            s_idx, t_idx = env.shift_idx[shift], env.team_idx[team]
            v = int(np.sum(np.maximum(0, demand[start:end, s_idx, t_idx] - cov[start:end, s_idx, t_idx])))
            per_pair.append(v)
            q_total += v
    total_viol += q_total
    print(f"Q{q} ({start:3d}-{end:3d})  " + " ".join(f"{v:>8}" for v in per_pair) + f" {q_total:>6}")
print("-" * len(header))
print(f"{'Total':<12} " + " ".join(f"{'':>8}" for _ in pair_labels) + f" {total_viol:>6}")

days_worked_arr = np.array(days_worked_list)
print(f"\nDays worked/employee: {days_worked_list}")
print(f"Mean: {days_worked_arr.mean():.1f}, Min: {days_worked_arr.min()}, Max: {days_worked_arr.max()}")
